# 🚗 Car Insurance Fraud Detection
### A Beginner-Friendly Machine Learning Project

---

## 📖 What is this project about?

Insurance companies lose billions of dollars every year because of **fraudulent claims** — people who fake or exaggerate accidents to get money they don't deserve.

Our goal is to build a **Machine Learning model** that can look at a claim and predict:
> *"Is this claim likely to be FRAUD or LEGITIMATE?"*

This is called a **Binary Classification** problem because the answer is one of two things: Fraud (Y) or Not Fraud (N).

---

## 🗺️ Project Roadmap

```
1. Load Data
2. Explore Data (EDA)
3. Preprocess Data (Clean + Encode + Scale)
4. Train Models
5. Evaluate Models
6. Compare & Choose the Best Model
7. Save the Best Model with Pickle  ← NEW
```

---
## Step 1: Import Libraries

| Library | Purpose |
|---|---|
| `pandas` | Load and manipulate data in tables (DataFrames) |
| `numpy` | Fast math and array operations |
| `matplotlib` / `seaborn` | Visualize data with charts |
| `sklearn` | The core ML library — models, preprocessing, evaluation |
| `pickle` | Save Python objects (like trained models) to files |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

print('✅ All libraries imported successfully!')

---
## Step 2: Load the Data

In [ ]:
data = pd.read_csv("/content/car_insurance_fraud_dataset.csv")

print(f"Dataset shape: {data.shape}")
print(f"  → {data.shape[0]} rows (claims)")
print(f"  → {data.shape[1]} columns (features)")
data.head()

---
## Step 3: Exploratory Data Analysis (EDA)

**EDA = Getting to know your data before building anything.**

> 💡 Most real ML work is EDA and preprocessing — NOT building models. Bad data = bad model.

In [ ]:
print("Column info:")
data.info()

In [ ]:
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found! Dataset is clean.")
else:
    print(missing_df)

In [ ]:
fraud_counts = data['fraud_reported'].value_counts()
fraud_pct    = data['fraud_reported'].value_counts(normalize=True) * 100

print("Fraud Distribution:")
print(pd.DataFrame({'Count': fraud_counts, 'Percentage (%)': fraud_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fraud_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Fraud vs Not Fraud (Count)', fontweight='bold')
axes[0].set_xlabel('N = Legit, Y = Fraud')
axes[0].tick_params(rotation=0)

axes[1].pie(fraud_counts, labels=['Not Fraud (N)', 'Fraud (Y)'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Fraud Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print("Numeric feature statistics:")
data.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Feature Patterns vs Fraud', fontsize=15, fontweight='bold')

sns.boxplot(data=data, x='fraud_reported', y='claim_amount',
            palette={'N': 'steelblue', 'Y': 'tomato'}, ax=axes[0, 0])
axes[0, 0].set_title('Claim Amount vs Fraud', fontweight='bold')

sns.histplot(data=data, x='insured_age', hue='fraud_reported',
             palette={'N': 'steelblue', 'Y': 'tomato'},
             kde=True, ax=axes[0, 1], element='step')
axes[0, 1].set_title('Age Distribution vs Fraud', fontweight='bold')

inc_fraud = data.groupby(['incident_type', 'fraud_reported']).size().unstack(fill_value=0)
inc_fraud.plot(kind='bar', ax=axes[1, 0], color=['steelblue', 'tomato'], edgecolor='black')
axes[1, 0].set_title('Incident Type vs Fraud', fontweight='bold')
axes[1, 0].tick_params(rotation=30)
axes[1, 0].legend(['Not Fraud', 'Fraud'])

sev_fraud = data.groupby(['incident_severity', 'fraud_reported']).size().unstack(fill_value=0)
sev_fraud.plot(kind='bar', ax=axes[1, 1], color=['steelblue', 'tomato'], edgecolor='black')
axes[1, 1].set_title('Incident Severity vs Fraud', fontweight='bold')
axes[1, 1].tick_params(rotation=15)
axes[1, 1].legend(['Not Fraud', 'Fraud'])

plt.tight_layout()
plt.show()

---
## Step 4: Data Preprocessing

| Problem | Solution |
|---|---|
| Non-useful columns (IDs, dates) | Drop them |
| Text columns (categories) | Encode as numbers with LabelEncoder |
| Numbers on very different scales | Normalize with StandardScaler |

In [ ]:
cols_to_drop = ['policy_id', 'incident_date', 'incident_city']
df = data.drop(columns=cols_to_drop)
print(f"Columns after dropping: {df.shape[1]} (was {data.shape[1]})")

In [ ]:
df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})
print("Target encoding:  Y → 1 (Fraud),  N → 0 (Legitimate)")
print(df['fraud_reported'].value_counts())

In [ ]:
# We need to track the exact label encodings used so the API can replicate them
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'fraud_reported']

# Store one LabelEncoder PER column (important for saving/loading later)
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le   # Save each encoder for later use

print(f"✅ Encoded {len(label_encoders)} categorical columns.")
print(f"Example — 'incident_severity' classes: {label_encoders['incident_severity'].classes_}")

In [ ]:
X = df.drop(columns=['fraud_reported'])
y = df['fraud_reported']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")

In [ ]:
# CRITICAL: fit ONLY on training data, then transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Scaling complete!")

---
## Step 5: Train Models

- **Logistic Regression** — Simple linear classifier, great baseline
- **Decision Tree** — Rule-based classifier, easy to interpret
- **Random Forest** — Ensemble of many trees, usually most accurate

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred       = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {'accuracy': acc, 'roc_auc': roc_auc,
                     'y_pred': y_pred, 'y_proba': y_pred_proba}

    print(f"{name:<25} Accuracy: {acc:.4f}  |  ROC-AUC: {roc_auc:.4f}")

print("\n✅ All models trained!")

---
## Step 6: Evaluate the Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Fraud', 'Fraud'],
                yticklabels=['Not Fraud', 'Fraud'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
for name, res in results.items():
    print(f"{'='*55}\n {name}\n{'='*55}")
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Not Fraud', 'Fraud']))

In [ ]:
plt.figure(figsize=(9, 6))
colors = ['steelblue', 'darkorange', 'green']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name}  (AUC = {res['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## Step 7: Compare & Feature Importance

In [ ]:
summary = pd.DataFrame({
    'Model':    list(results.keys()),
    'Accuracy': [res['accuracy'] for res in results.values()],
    'ROC-AUC':  [res['roc_auc']  for res in results.values()]
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("📊 Model Comparison:")
print(summary.to_string(index=False))

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top_features = importances.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features.index[::-1], top_features.values[::-1], color='steelblue', edgecolor='black')
plt.xlabel('Importance Score')
plt.title('Top 15 Most Important Features (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8: Save the Best Model with Pickle 💾

### What is Pickle?

**Pickle** is a Python built-in library that can **serialize** any Python object — meaning it converts it into a binary file that can be saved to disk and loaded back later.

Think of it like this:
> You spend hours training a model. Without saving it, you'd have to retrain it every single time you want to use it. Pickle lets you save the trained model once and load it instantly anytime.

### What do we need to save?

We need to save **3 things** — not just the model:

| File | Why? |
|---|---|
| `model.pkl` | The trained Random Forest (its learned rules) |
| `scaler.pkl` | The scaler fitted on training data (same mean/std must be used on new data) |
| `label_encoders.pkl` | The encoders for each text column (same mapping must be used on new data) |

> ⚠️ If you don't save the scaler and encoders, new data will be transformed differently and the model will give wrong predictions!

In [ ]:
# ── Determine the best model automatically (highest ROC-AUC) ──────────────────
best_model_name = max(results, key=lambda name: results[name]['roc_auc'])
best_model      = models[best_model_name]

print(f"🏆 Best model: {best_model_name}")
print(f"   ROC-AUC: {results[best_model_name]['roc_auc']:.4f}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.4f}")

In [ ]:
# ── Create a folder to store the saved files ──────────────────────────────────
os.makedirs('/content/model_artifacts', exist_ok=True)

# ── Save the trained model ────────────────────────────────────────────────────
# 'wb' = write binary (pickle files are binary, not plain text)
with open('/content/model_artifacts/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# ── Save the scaler ───────────────────────────────────────────────────────────
with open('/content/model_artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# ── Save all label encoders (as a dictionary) ─────────────────────────────────
with open('/content/model_artifacts/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# ── Save the feature column names (needed to align input order) ───────────────
with open('/content/model_artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print("✅ Saved successfully to /content/model_artifacts/")
print("   → model.pkl")
print("   → scaler.pkl")
print("   → label_encoders.pkl")
print("   → feature_columns.pkl")

In [ ]:
# ── Verify: Load the model back and test it ───────────────────────────────────
# 'rb' = read binary
with open('/content/model_artifacts/model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('/content/model_artifacts/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

# Test the loaded model on the test set — results should match original
y_pred_loaded = loaded_model.predict(loaded_scaler.transform(X_test))
acc_loaded    = accuracy_score(y_test, y_pred_loaded)

print(f"✅ Loaded model accuracy: {acc_loaded:.4f}")
print(f"   Original model accuracy: {results[best_model_name]['accuracy']:.4f}")
print("   → Both match! The model was saved and loaded correctly.")

In [ ]:
# ── Download the files from Colab to your local machine ──────────────────────
# Run this cell to download all 4 artifact files
from google.colab import files

for fname in ['model.pkl', 'scaler.pkl', 'label_encoders.pkl', 'feature_columns.pkl']:
    files.download(f'/content/model_artifacts/{fname}')
    print(f"⬇️  Downloading {fname}...")

---
## 📝 Final Summary

| Concept | What It Means |
|---|---|
| **EDA** | Exploring data before modeling to understand its patterns |
| **Label Encoding** | Converting text categories to numbers |
| **Train/Test Split** | Separating data so we test on unseen examples |
| **StandardScaler** | Normalizing feature scales so no feature dominates |
| **Logistic Regression** | Simple linear classifier — great baseline |
| **Decision Tree** | Rule-based classifier — easy to interpret |
| **Random Forest** | Ensemble of trees — usually most accurate |
| **Pickle** | Saves trained Python objects (models, scalers) to disk |
| **ROC-AUC** | Best metric for imbalanced classification problems |
| **Feature Importance** | Which input features helped the model most |

### Next Steps
- **Hyperparameter Tuning** with `GridSearchCV`
- **SMOTE** for handling imbalanced classes
- **Cross-Validation** for more robust evaluation
- **XGBoost / LightGBM** — industry-standard tree models